In [4]:
!pip install google-generativeai -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:

from google import genai
from google.genai import types

client = genai.Client()

# A list of dictionaries, where each is a GenerateContentRequest
inline_requests = [
    {
        'contents': [{
            'parts': [{'text': 'Tell me a one-sentence joke.'}],
            'role': 'user'
        }]
    },
    {
        'contents': [{
            'parts': [{'text': 'Why is the sky blue?'}],
            'role': 'user'
        }]
    }
]

inline_batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=inline_requests,
    config={
        'display_name': "inlined-requests-job-1",
    },
)

print(f"Created batch job: {inline_batch_job.name}")

Created batch job: batches/rthsn186okepqkxh7habwjjxjxmjst7vkrgf


In [27]:
# 獲取任務目前的最新資訊
job = client.batches.get(name="batches/rthsn186okepqkxh7habwjjxjxmjst7vkrgf")
# print(job)

print(f"目前任務狀態: {job.state}")

目前任務狀態: JobState.JOB_STATE_SUCCEEDED


In [28]:
import json
from google import genai

client = genai.Client()

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = inline_batch_job.name  # Replace with your actual job name if not using the variable
batch_job = client.batches.get(name=job_name)

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

    # If batch job was created with a file
    if batch_job.dest and batch_job.dest.file_name:
        # Results are in a file
        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("Downloading result file content...")
        file_content = client.files.download(file=result_file_name)
        # Process file_content (bytes) as needed
        print(file_content.decode('utf-8'))

    # If batch job was created with inline request
    # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
    elif batch_job.dest and batch_job.dest.inlined_responses:
        # Results are inline
        print("Results are inline:")
        for i, inline_response in enumerate(batch_job.dest.inlined_responses):
            print(f"Response {i+1}:")
            if inline_response.response:
                # Accessing response, structure may vary.
                try:
                    print(inline_response.response.text)
                except AttributeError:
                    print(inline_response.response) # Fallback
            elif inline_response.error:
                print(f"Error: {inline_response.error}")
    else:
        print("No results found (neither file nor inline).")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")
    if batch_job.error:
        print(f"Error: {batch_job.error}")

Results are inline:
Response 1:
I told my wife she was drawing her eyebrows too high, and she looked surprised.
Response 2:
The short answer is a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength. Red light has long, lazy waves, while blue and violet light have much shorter, choppier waves.

### 2. The Atmosphere is an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). When sunlight strikes the atmosphere, it collides with the molecules of these gases.

### 3. Blue waves scatter more easily
Because blue light travels in shorter, smaller waves, it strikes the gas molecules and gets scattered in every direction. This is called **Rayleigh scattering**. 

Beca

In [30]:
pip install pandas -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import pandas as pd

# 讀取資料（假設是 CSV）
df = pd.read_csv("Reviews_withURL.csv")

# 隨機抽樣 5000 筆
sample_df = df.sample(n=5000, random_state=42)

# ⭐ 清理 Text（避免 NaN）
sample_df["Text"] = sample_df["Text"].fillna("").astype(str)

# 只保留指定欄位
sample_df = sample_df[["Id", "ProductId", "Text"]]

print("抽樣完成:", sample_df.shape)
print(sample_df.head())

抽樣完成: (5000, 3)
            Id   ProductId                                               Text
165256  165257  B000EVG8J2  Having tried a couple of other brands of glute...
231465  231466  B0000BXJIS  My cat loves these treats. If ever I can't fin...
427827  427828  B008FHUFAU  A little less than I expected.  It tends to ha...
433954  433955  B006BXV14E  First there was Frosted Mini-Wheats, in origin...
70260    70261  B007I7Z3Z0  and I want to congratulate the graphic artist ...


In [40]:
sample_reviews = sample_df[["Id", "Text"]].to_dict("records")
display(sample_reviews[:5])

[{'Id': 165257,
  'Text': 'Having tried a couple of other brands of gluten-free sandwich cookies, these are the best of the bunch.  They\'re crunchy and true to the texture of the other "real" cookies that aren\'t gluten-free.  Some might think that the filling makes them a bit too sweet, but for me that just means I\'ve satisfied my sweet tooth sooner!  The chocolate version from Glutino is just as good and has a true "chocolatey" taste - something that isn\'t there with the other gluten-free brands out there.'},
 {'Id': 231466,
  'Text': "My cat loves these treats. If ever I can't find her in the house, I just pop the top and she bolts out of wherever she was hiding to come get a treat. She doesn't like crunchy treats much, so these are perfect for her. I've given her all three flavors and she seems to like them all equally. They do tend to dry out by the time I near the end of the bottle, however. The flip-top lid is very handy. Very nice, inexpensive kitty treats. I have yet to mee

預先側兩筆

In [36]:
from google import genai
import os
import time
import json
import pandas as pd

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# 測試資料
reviews = [
    {
        "Id": 1,
        "Text": "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than most."
    },
    {
        "Id": 2,
        "Text": "Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as \"Jumbo\"."
    }
]

def build_prompt(review_id, text):
    return f"""
You are a food safety review classifier.

Classify the review into exactly ONE label from:
- contamination
- spoilage
- illness
- allergen
- quality_defect
- safe

Definitions:
- contamination: foreign objects or harmful substances such as plastic, metal, glass, insects, worms, chemicals
- spoilage: spoiled, rotten, rancid, moldy, expired, bad smell due to spoilage
- illness: the person or pet became sick after consuming the product
- allergen: allergic reaction or unexpected allergen exposure
- quality_defect: non-safety product issue such as bad texture, stale product, crushed packaging, melted item, wrong product, wrong label, misleading packaging, etc.
- safe: no safety or defect issue is described

Important rules:
- Return exactly one label.
- Return valid JSON only.
- Do not output markdown.

Output format:
{{
  "Id": {review_id},
  "label": "one_label_only"
}}

Review:
Id: {review_id}
Text: {text}
"""

# 建立 inline requests
inline_requests = []
for row in reviews:
    inline_requests.append({
        "contents": [
            {
                "role": "user",
                "parts": [{"text": build_prompt(row["Id"], row["Text"])}]
            }
        ],
        "config": {
            "temperature": 0,
            "response_mime_type": "application/json"
        }
    })

# 建立 Batch job
batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=inline_requests,
    config={
        "display_name": "food-safety-batch"
    }
)

print("Batch job created:", batch_job.name)
print("Initial state:", batch_job.state)

# 輪詢直到完成
job_name = batch_job.name

while True:
    job = client.batches.get(name=job_name)
    state = job.state.name if hasattr(job.state, "name") else str(job.state)
    print("Current state:", state)

    if state in [
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
        "JOB_STATE_EXPIRED"
    ]:
        break

    time.sleep(5)

# 解析結果
results = []

if state == "JOB_STATE_SUCCEEDED":
    for res in job.dest.inlined_responses:
        try:
            text = res.response.text.strip()
            data = json.loads(text)

            results.append({
                "Id": data["Id"],
                "label": data["label"]
            })
        except Exception as e:
            results.append({
                "Id": None,
                "label": "ERROR"
            })

    result_df = pd.DataFrame(results)

    # 可選：按 Id 排序，方便看
    result_df = result_df.sort_values("Id").reset_index(drop=True)

    print(result_df)

    # 輸出 CSV
    result_df.to_csv("food_safety_labels.csv", index=False, encoding="utf-8-sig")
    print("CSV 已輸出：food_safety_labels.csv")
else:
    print("Batch job did not succeed.")

Batch job created: batches/sv2ypii7px6ci77plrfzr6hzwng2nw4iqghz
Initial state: JobState.JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Cur

正式執行

In [41]:
import pandas as pd
from google import genai
import os
import time
import json



# =========================
# 2. Gemini client
# =========================
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# 直接用 sample_reviews
reviews = sample_reviews

def build_prompt(review_id, text):
    return f"""
You are a food safety review classifier.

Classify the review into exactly ONE label from:
- contamination
- spoilage
- illness
- allergen
- quality_defect
- safe

Definitions:
- contamination: foreign objects or harmful substances such as plastic, metal, glass, insects, worms, chemicals
- spoilage: spoiled, rotten, rancid, moldy, expired, bad smell due to spoilage
- illness: the person or pet became sick after consuming the product
- allergen: allergic reaction or unexpected allergen exposure
- quality_defect: non-safety product issue such as bad texture, stale product, crushed packaging, melted item, wrong product, wrong label, misleading packaging, etc.
- safe: no safety or defect issue is described

Important rules:
- Return exactly one label.
- Return valid JSON only.
- Do not output markdown.

Output format:
{{
  "Id": {review_id},
  "label": "one_label_only"
}}

Review:
Id: {review_id}
Text: {text}
"""

# =========================
# 3. 建立 inline requests
# =========================
inline_requests = []
for row in reviews:
    inline_requests.append({
        "contents": [
            {
                "role": "user",
                "parts": [{"text": build_prompt(row["Id"], row["Text"])}]
            }
        ],
        "config": {
            "temperature": 0,
            "response_mime_type": "application/json"
        }
    })

# =========================
# 4. 建立 Batch job
# =========================
batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src=inline_requests,
    config={
        "display_name": "food-safety-batch"
    }
)

print("Batch job created:", batch_job.name)
print("Initial state:", batch_job.state)

# =========================
# 5. 輪詢直到完成
# =========================
job_name = batch_job.name

while True:
    job = client.batches.get(name=job_name)
    state = job.state.name if hasattr(job.state, "name") else str(job.state)
    print("Current state:", state)

    if state in [
        "JOB_STATE_SUCCEEDED",
        "JOB_STATE_FAILED",
        "JOB_STATE_CANCELLED",
        "JOB_STATE_EXPIRED"
    ]:
        break

    time.sleep(5)

# =========================
# 6. 解析結果
# =========================
results = []

if state == "JOB_STATE_SUCCEEDED":
    for res in job.dest.inlined_responses:
        try:
            text = res.response.text.strip()
            data = json.loads(text)

            results.append({
                "Id": data["Id"],
                "label": data["label"]
            })
        except Exception:
            results.append({
                "Id": None,
                "label": "ERROR"
            })

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("Id").reset_index(drop=True)

    print(result_df.head())

    result_df.to_csv("food_safety_labels.csv", index=False, encoding="utf-8-sig")
    print("CSV 已輸出：food_safety_labels.csv")

else:
    print("Batch job did not succeed.")

Batch job created: batches/xg5h3kqtomijwqegtmg8kszjuf3p9ge2ptv6
Initial state: JobState.JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_PENDING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Cur

In [45]:
import pandas as pd

# 讀取 label 結果
label_df = pd.read_csv("food_safety_labels.csv")

# 合併回原資料
final_df = sample_df.merge(label_df, on="Id", how="left")
# 新增欄位 human_is_hazard
final_df["human_is_hazard"] = final_df["label"].apply(lambda x: 0 if x == "safe" else 1)
final_df.to_csv("sample_with_labels.csv", index=False, encoding="utf-8-sig")

print(final_df.head())

       Id   ProductId                                               Text  \
0  165257  B000EVG8J2  Having tried a couple of other brands of glute...   
1  231466  B0000BXJIS  My cat loves these treats. If ever I can't fin...   
2  427828  B008FHUFAU  A little less than I expected.  It tends to ha...   
3  433955  B006BXV14E  First there was Frosted Mini-Wheats, in origin...   
4   70261  B007I7Z3Z0  and I want to congratulate the graphic artist ...   

            label  human_is_hazard  
0            safe                0  
1            safe                0  
2  quality_defect                1  
3            safe                0  
4            safe                0  
